# Notebook 08 — Visual Generation Tool Layer

**Architecture:** v2.2 — SchemDraw + Kroki + local structured renderer with fail-closed integrity gating.

Notebook 08 still consumes the validated visual handoff from Notebook 06 and renders required visuals.

### v2.2 sync note

Notebook 06 v2.38 now performs stronger upstream checks for:
- broader missing-visual reference detection;
- visual compatibility;
- cross-pattern duplicate detection;
- visual relevance/dependency;
- generic entity consistency.

Notebook 08 remains the canonical renderer/integrity layer. If a required visual fails here, the patched manifest is explicitly not release-ready.


## 1. Final renderer policy

| Visual type | Backend | Engine | Future MCP tool |
|---|---|---|---|
| `logic_gate_diagram` | **SchemDraw** | logic gates | `render_logic_visual` |
| `network_diagram` | **Kroki** | GraphViz | `render_technical_visual` |
| `simple_flowchart` | **Kroki** | GraphViz | `render_technical_visual` |
| `cpu_block_diagram` | **Kroki** | GraphViz | `render_technical_visual` |
| `truth_table` | local structured renderer | table | `render_structured_visual` |
| `code_block` | local structured renderer | code | `render_structured_visual` |
| `trace_table` | local structured renderer | table | `render_structured_visual` |
| `array_grid` | local structured renderer | grid | `render_structured_visual` |
| `database_table` | local structured renderer | table | `render_structured_visual` |
| `memory_grid` | local structured renderer | grid | `render_structured_visual` |
| `binary_register` | local structured renderer | grid | `render_structured_visual` |

### Deliberate removal

Notebook 08 no longer requires:
- the Python `graphviz` package;
- a local Graphviz `dot` executable;
- hand-positioned matplotlib network/flowchart/CPU fallback diagrams.

Those technical diagrams are sent to Kroki. The local renderer is kept only for
structured assessment visuals where its current output is already appropriate.


In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import re
import subprocess
import sys
import urllib.error
import urllib.request

from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from IPython.display import display, Image as IPyImage


# ================================================================
# PATHS
# ================================================================

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name.casefold() in {"notebooks", "notebook"}:
    AGENT2_ROOT = CURRENT_DIR.parent
else:
    AGENT2_ROOT = Path(
        os.getenv("AGENT2_PROJECT_ROOT", str(CURRENT_DIR))
    ).expanduser().resolve()

NOTEBOOK06_OUTPUT_DIR = Path(
    os.getenv(
        "AGENT2_QUIZ_OUTPUT_DIR",
        str(AGENT2_ROOT / "OUTPUT" / "notebook_06_quiz"),
    )
).expanduser().resolve()

OUTPUT_DIR = Path(
    os.getenv(
        "AGENT2_VISUAL_OUTPUT_DIR",
        str(AGENT2_ROOT / "OUTPUT" / "notebook_08_visuals"),
    )
).expanduser().resolve()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VISUAL_ASSET_DIR = OUTPUT_DIR / "assets"
VISUAL_ASSET_DIR.mkdir(parents=True, exist_ok=True)

HANDOFF_PATH = Path(
    os.getenv(
        "AGENT2_VISUAL_HANDOFF_PATH",
        str(NOTEBOOK06_OUTPUT_DIR / "visual_tool_handoff.json"),
    )
).expanduser().resolve()

RESULTS_PATH = OUTPUT_DIR / "notebook08_visual_results.json"
UPDATED_MANIFEST_PATH = (
    OUTPUT_DIR
    / "final_quiz_manifest_with_notebook08_visuals.json"
)

PATCH_NOTEBOOK06_MANIFEST = str(
    os.getenv(
        "AGENT2_VISUAL_PATCH_NOTEBOOK06_MANIFEST",
        "1",
    )
).strip().casefold() in {"1", "true", "yes", "on"}

RUN_SMOKE_TESTS = str(
    os.getenv(
        "AGENT2_VISUAL_RUN_SMOKE_TESTS",
        "0",
    )
).strip().casefold() in {"1", "true", "yes", "on"}


# ================================================================
# KROKI CONFIG
# ================================================================
# Development default uses the public Kroki endpoint.
# For production, set AGENT2_KROKI_ENDPOINT to your self-hosted instance.
#
# Examples:
#   AGENT2_KROKI_ENDPOINT=https://kroki.io
#   AGENT2_KROKI_ENDPOINT=http://localhost:8000
# ================================================================

KROKI_ENDPOINT = str(
    os.getenv(
        "AGENT2_KROKI_ENDPOINT",
        "https://kroki.io",
    )
).strip().rstrip("/")

KROKI_TIMEOUT_SECONDS = float(
    os.getenv(
        "AGENT2_KROKI_TIMEOUT_SECONDS",
        "20",
    )
)

KROKI_OUTPUT_FORMAT = "png"

KROKI_ENDPOINT_IS_PUBLIC = (
    "kroki.io" in KROKI_ENDPOINT.casefold()
)


# ================================================================
# LIGHTWEIGHT DEPENDENCY BOOTSTRAP
# ================================================================

def ensure_package(
    import_name: str,
    pip_name: str | None = None,
) -> None:
    if importlib.util.find_spec(import_name) is not None:
        return

    package_name = pip_name or import_name
    print(f"Installing missing package: {package_name}")

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            package_name,
        ]
    )


ensure_package("schemdraw")
ensure_package("matplotlib")

import matplotlib
matplotlib.use("Agg", force=True)

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import schemdraw
from schemdraw.parsing import logicparse


print("Notebook 06 handoff:", HANDOFF_PATH)
print("Notebook 08 output:", OUTPUT_DIR)
print("Kroki endpoint:", KROKI_ENDPOINT)
print("Kroki timeout:", KROKI_TIMEOUT_SECONDS, "seconds")

if KROKI_ENDPOINT_IS_PUBLIC:
    print(
        "Kroki mode: public endpoint (development/testing). "
        "Use a self-hosted endpoint for production."
    )
else:
    print("Kroki mode: custom/self-hosted endpoint.")


### Kroki configuration

Notebook 08 sends technical diagram source text to a configurable Kroki endpoint.

Default development setting:

```text
AGENT2_KROKI_ENDPOINT=https://kroki.io
```

Production should point the same variable at the project's self-hosted Kroki
instance, for example:

```text
AGENT2_KROKI_ENDPOINT=http://localhost:8000
```

The current implementation uses Kroki's **GraphViz engine** for network,
flowchart and CPU/block diagrams. This avoids requiring a separate local
Graphviz installation while keeping automatic graph layout.

If Kroki is unavailable for a required technical diagram, rendering fails
closed for that asset. Notebook 08 does **not** fall back to hand-positioned
technical diagrams.


In [ ]:
# ================================================================
# SHARED HELPERS
# ================================================================

VISUAL_SCHEMA_VERSION = "agent2-visual-architecture-v3.0.0"
NOTEBOOK08_RESULTS_SCHEMA_VERSION = "agent2-notebook08-visual-results-v1.1.0"

VISUAL_TYPES = {
    "none",
    "code_block",
    "trace_table",
    "array_grid",
    "simple_flowchart",
    "logic_gate_diagram",
    "truth_table",
    "network_diagram",
    "database_table",
    "cpu_block_diagram",
    "memory_grid",
    "binary_register",
}


def normalize_visual_type(value: Any) -> str:
    normalized = str(value or "none").strip().casefold()

    aliases = {
        "": "none",
        "no": "none",
        "false": "none",
        "flowchart": "simple_flowchart",
        "flow_chart": "simple_flowchart",
        "array": "array_grid",
        "table": "trace_table",
        "code": "code_block",
        "logic": "logic_gate_diagram",
        "logic_gate": "logic_gate_diagram",
        "logic_gates": "logic_gate_diagram",
        "truth": "truth_table",
        "network": "network_diagram",
        "database": "database_table",
        "cpu": "cpu_block_diagram",
        "memory": "memory_grid",
        "binary": "binary_register",
        "bit_register": "binary_register",
    }

    normalized = aliases.get(normalized, normalized)

    if normalized not in VISUAL_TYPES:
        raise ValueError(
            f"Unsupported visual type: {value!r}"
        )

    return normalized


def stable_json_fingerprint(value: Any) -> str:
    canonical = json.dumps(
        value,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
        default=str,
    )
    return hashlib.sha256(
        canonical.encode("utf-8")
    ).hexdigest()


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def safe_identifier(value: Any, fallback: str = "visual") -> str:
    cleaned = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        str(value or ""),
    ).strip("_")

    return cleaned or fallback


def asset_path(
    question: dict[str, Any],
    visual_type: str,
    extension: str = "png",
) -> Path:
    qid = safe_identifier(
        question.get(
            "generated_question_id",
            question.get("question_index", "visual"),
        )
    )

    spec_hash = stable_json_fingerprint(
        question.get("visual", {})
    )[:12]

    return (
        VISUAL_ASSET_DIR
        / f"{qid}_{visual_type}_{spec_hash}.{extension}"
    )


def save_matplotlib_figure(
    fig: Any,
    path: Path,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    fig.savefig(
        path,
        dpi=180,
        bbox_inches="tight",
        facecolor="white",
    )

    plt.close(fig)


## 2. Load Notebook 06 handoff

Notebook 08 expects the `visual_tool_handoff.json` produced by Notebook 06 v2.34.

No transcript, syllabus, question bank, or LLM prompt is loaded again here. This keeps the visual layer focused and prevents duplicate model calls.


In [ ]:
if not HANDOFF_PATH.is_file():
    raise FileNotFoundError(
        "Notebook 08 could not find the Notebook 06 visual handoff.\n"
        f"Expected: {HANDOFF_PATH}\n"
        "Run Notebook 06 v2.34 first, or set AGENT2_VISUAL_HANDOFF_PATH."
    )

handoff = json.loads(
    HANDOFF_PATH.read_text(
        encoding="utf-8"
    )
)

handoff_questions = handoff.get(
    "questions",
    [],
)

if not isinstance(
    handoff_questions,
    list,
):
    raise ValueError(
        "visual_tool_handoff.json field 'questions' must be a list."
    )

print(
    "Handoff schema:",
    handoff.get("schema_version"),
)

print(
    "Questions in handoff:",
    len(handoff_questions),
)

print(
    "Questions requiring visuals:",
    sum(
        1
        for question in handoff_questions
        if normalize_visual_type(
            question.get(
                "visual_requirement",
                "none",
            )
        )
        != "none"
    ),
)


## 3. Visual contract validation

Notebook 08 validates the renderer-facing subset again before creating an asset. This is intentionally cheap and deterministic.

The validation here does **not** replace Notebook 06 validation. It protects the rendering boundary from malformed handoff data.


In [ ]:
_VISUAL_REFERENCE_PATTERNS_NB08 = [
    r"\blook at (?:the|this) (?:diagram|visual|grid|table|flowchart)\b",
    r"\brefer to (?:the|this) (?:diagram|visual|grid|table|flowchart)\b",
    r"\bshown in (?:the|this) (?:diagram|visual|grid|table|flowchart)\b",
    r"\b(?:diagram|visual|grid|flowchart) (?:shown|provided|below|above)\b",
    r"\bthe network diagram\b",
    r"\bthe logic gate diagram\b",
]


def _nb08_question_references_visual(
    question_text_value: Any,
) -> bool:
    text_value = str(
        question_text_value or ""
    ).casefold()

    return any(
        re.search(
            pattern,
            text_value,
            flags=re.IGNORECASE,
        )
        is not None
        for pattern in _VISUAL_REFERENCE_PATTERNS_NB08
    )

def validate_visual_question(
    question: dict[str, Any],
) -> list[str]:
    errors: list[str] = []

    try:
        visual_type = normalize_visual_type(
            question.get(
                "visual_requirement",
                "none",
            )
        )
    except Exception as exc:
        return [
            str(exc)
        ]

    if (
        visual_type == "none"
        and _nb08_question_references_visual(
            question.get(
                "question_text",
                "",
            )
        )
    ):
        errors.append(
            "question_text refers to a visual but visual_requirement='none'"
        )

    visual = question.get(
        "visual",
        {},
    )

    if not isinstance(
        visual,
        dict,
    ):
        return errors + [
            "visual must be a dictionary"
        ]

    spec = visual.get(
        "spec",
        {},
    )

    if not isinstance(
        spec,
        dict,
    ):
        return errors + [
            "visual.spec must be a dictionary"
        ]

    if visual_type == "none":
        return errors

    if visual_type == "logic_gate_diagram":
        inputs = spec.get(
            "inputs",
            [],
        )
        gates = spec.get(
            "gates",
            [],
        )
        output = spec.get(
            "output",
            {},
        )

        if not isinstance(
            inputs,
            list,
        ) or not inputs:
            errors.append(
                "logic_gate_diagram requires a non-empty inputs list"
            )

        if not isinstance(
            gates,
            list,
        ) or not gates:
            errors.append(
                "logic_gate_diagram requires a non-empty gates list"
            )

        if not isinstance(
            output,
            dict,
        ):
            errors.append(
                "logic_gate_diagram output must be a dictionary"
            )

    elif visual_type in {
        "truth_table",
        "trace_table",
        "database_table",
    }:
        columns = spec.get(
            "columns",
            [],
        )
        rows = spec.get(
            "rows",
            [],
        )

        if not isinstance(
            columns,
            list,
        ) or not columns:
            errors.append(
                f"{visual_type} requires columns"
            )

        if not isinstance(
            rows,
            list,
        ):
            errors.append(
                f"{visual_type} rows must be a list"
            )

    elif visual_type == "array_grid":
        values = spec.get(
            "values",
            [],
        )

        if not isinstance(
            values,
            list,
        ) or not values:
            errors.append(
                "array_grid requires values"
            )

    elif visual_type in {
        "simple_flowchart",
        "network_diagram",
    }:
        nodes = spec.get(
            "nodes",
            [],
        )
        edges = spec.get(
            "edges",
            [],
        )

        if not isinstance(
            nodes,
            list,
        ) or not nodes:
            errors.append(
                f"{visual_type} requires nodes"
            )

        if not isinstance(
            edges,
            list,
        ):
            errors.append(
                f"{visual_type} edges must be a list"
            )

    elif visual_type == "cpu_block_diagram":
        components = spec.get(
            "components",
            [],
        )
        connections = spec.get(
            "connections",
            [],
        )

        if (
            not isinstance(
                components,
                list,
            )
            or not components
        ):
            errors.append(
                "cpu_block_diagram requires components"
            )

        if not isinstance(
            connections,
            list,
        ):
            errors.append(
                "cpu_block_diagram connections must be a list"
            )

    elif visual_type == "memory_grid":
        addresses = spec.get(
            "addresses",
            [],
        )
        values = spec.get(
            "values",
            [],
        )

        if (
            not isinstance(
                addresses,
                list,
            )
            or not addresses
        ):
            errors.append(
                "memory_grid requires addresses"
            )

        if (
            not isinstance(
                values,
                list,
            )
            or len(values)
            != len(addresses)
        ):
            errors.append(
                "memory_grid values must match addresses"
            )

    elif visual_type == "binary_register":
        bits = spec.get(
            "bits",
            [],
        )

        if (
            not isinstance(
                bits,
                list,
            )
            or not bits
        ):
            errors.append(
                "binary_register requires bits"
            )

    elif visual_type == "code_block":
        code = str(
            spec.get(
                "code",
                "",
            )
            or ""
        )

        if not code.strip():
            errors.append(
                "code_block requires non-empty code"
            )

    return errors

## 4. SchemDraw renderer — Boolean / logic circuits

Logic circuits use the specialist SchemDraw backend so AND, OR, NOT, NAND, NOR,
XOR and XNOR structures are represented with proper digital-logic symbols.

Truth tables are intentionally handled by the local structured renderer because
they are exact-value assessment tables rather than free-form circuit diagrams.


In [ ]:
SUPPORTED_LOGIC_GATES = {
    "AND",
    "OR",
    "NOT",
    "NAND",
    "NOR",
    "XOR",
    "XNOR",
}


def _logic_expression_from_spec(
    spec: dict[str, Any],
) -> tuple[str, str]:
    input_names = [
        str(value)
        for value in spec.get(
            "inputs",
            [],
        )
    ]

    gate_map: dict[str, dict[str, Any]] = {}

    for gate in spec.get(
        "gates",
        [],
    ):
        if not isinstance(
            gate,
            dict,
        ):
            continue

        gate_id = str(
            gate.get(
                "id",
                "",
            )
            or ""
        ).strip()

        if gate_id:
            gate_map[
                gate_id
            ] = gate

    active_stack: set[str] = set()

    def resolve(
        ref: str,
    ) -> str:
        if ref in input_names:
            return ref

        if ref not in gate_map:
            return ref

        if ref in active_stack:
            raise ValueError(
                "Cycle detected in logic-gate specification."
            )

        active_stack.add(
            ref
        )

        gate = gate_map[
            ref
        ]

        gate_type = str(
            gate.get(
                "type",
                "",
            )
            or ""
        ).strip().upper()

        if gate_type not in SUPPORTED_LOGIC_GATES:
            raise ValueError(
                f"Unsupported logic gate: {gate_type}"
            )

        refs = [
            str(value)
            for value in gate.get(
                "inputs",
                [],
            )
        ]

        child_exprs = [
            resolve(
                child_ref
            )
            for child_ref in refs
        ]

        if gate_type == "NOT":
            if len(child_exprs) != 1:
                raise ValueError(
                    "NOT gate must have exactly one input."
                )

            expression = (
                f"not ({child_exprs[0]})"
            )

        elif gate_type in {
            "AND",
            "OR",
            "XOR",
        }:
            if len(child_exprs) < 2:
                raise ValueError(
                    f"{gate_type} gate must have at least two inputs."
                )

            operator = gate_type.casefold()

            expression = (
                "("
                + f" {operator} ".join(
                    child_exprs
                )
                + ")"
            )

        elif gate_type in {
            "NAND",
            "NOR",
            "XNOR",
        }:
            if len(child_exprs) < 2:
                raise ValueError(
                    f"{gate_type} gate must have at least two inputs."
                )

            base_operator = {
                "NAND": "and",
                "NOR": "or",
                "XNOR": "xor",
            }[
                gate_type
            ]

            expression = (
                "not ("
                + f" {base_operator} ".join(
                    child_exprs
                )
                + ")"
            )

        else:
            raise ValueError(
                f"Unhandled logic gate: {gate_type}"
            )

        active_stack.remove(
            ref
        )

        return expression

    output = spec.get(
        "output",
        {},
    )

    if not isinstance(
        output,
        dict,
    ):
        output = {}

    root_ref = str(
        output.get(
            "from",
            "",
        )
        or ""
    ).strip()

    if not root_ref:
        if not gate_map:
            raise ValueError(
                "No logic-gate output could be resolved."
            )

        root_ref = list(
            gate_map.keys()
        )[-1]

    output_label = str(
        output.get(
            "label",
            "Q",
        )
        or "Q"
    ).strip()

    return (
        resolve(root_ref),
        output_label,
    )


def _logic_gate_labels_for_render(
    question: dict[str, Any],
    spec: dict[str, Any],
) -> list[str]:
    """
    Resolve learner-facing gate-position labels without assuming question
    numbers, gate counts or particular label values.
    """
    gates = spec.get(
        "gates",
        [],
    )

    if not isinstance(
        gates,
        list,
    ):
        return []

    labels = [
        str(
            gate.get(
                "label",
                "",
            )
            or ""
        ).strip()
        if isinstance(
            gate,
            dict,
        )
        else ""
        for gate in gates
    ]

    if (
        labels
        and all(labels)
    ):
        return labels

    handoff_labels = question.get(
        "logic_gate_position_labels",
        [],
    )

    if (
        isinstance(
            handoff_labels,
            list,
        )
        and len(
            handoff_labels
        )
        == len(gates)
        and all(
            str(value or "").strip()
            for value in handoff_labels
        )
    ):
        return [
            str(value).strip()
            for value in handoff_labels
        ]

    # Final generic fallback: if the learner-facing question explicitly
    # refers to labelled gate positions/boxes, recover those identifiers
    # from the question text. This avoids relying on a particular handoff
    # field while still requiring an exact one-label-per-spec-gate match.
    question_text = str(
        question.get(
            "question_text",
            "",
        )
        or ""
    )

    if re.search(
        r"\b(?:label(?:led|ed)?|position|positions|box|boxes)\b",
        question_text,
        flags=re.IGNORECASE,
    ):
        recovered_labels: list[str] = []

        for match in re.findall(
            r"\b[A-Za-z][A-Za-z_-]*\d+\b",
            question_text,
        ):
            label = str(match).strip()

            if (
                label
                and label not in recovered_labels
            ):
                recovered_labels.append(
                    label
                )

        if (
            len(recovered_labels)
            == len(gates)
            and all(recovered_labels)
        ):
            return recovered_labels

    return []


def _attach_logic_gate_position_labels(
    drawing: Any,
    labels: list[str],
    spec: dict[str, Any],
) -> bool:
    """
    Add learner-facing gate-position identifiers to the SchemDraw circuit.

    SchemDraw's logic parser may collapse an explicit base gate followed by
    a NOT gate into one compound symbol (for example OR -> NOT becomes NOR).
    When that happens, keep both original position labels visible on that
    compound symbol instead of dropping all labels. No question numbers,
    label values, gate counts or specific circuit shapes are hardcoded.
    """
    if not labels:
        return False

    gate_class_tokens = {
        "and",
        "or",
        "not",
        "nand",
        "nor",
        "xor",
        "xnor",
    }

    gate_elements: list[tuple[Any, str]] = []

    for element in getattr(
        drawing,
        "elements",
        [],
    ):
        class_name = (
            element.__class__.__name__
            .strip()
            .casefold()
        )

        matched_type = ""
        if class_name in gate_class_tokens:
            matched_type = class_name
        else:
            # Schemdraw class names can contain a small suffix/prefix
            # depending on version. Keep matching limited to known gates.
            for token in sorted(
                gate_class_tokens,
                key=len,
                reverse=True,
            ):
                if (
                    class_name.startswith(token)
                    or class_name.endswith(token)
                ):
                    matched_type = token
                    break

        if matched_type:
            gate_elements.append(
                (element, matched_type)
            )

    def add_label(
        element: Any,
        label: str,
        *,
        loc: str,
    ) -> None:
        try:
            element.label(
                label,
                loc=loc,
            )
        except TypeError:
            element.label(
                label
            )

    # Normal case: one explicit specification gate maps to one rendered gate.
    if len(gate_elements) == len(labels):
        for (element, _), label in zip(
            gate_elements,
            labels,
        ):
            add_label(
                element,
                label,
                loc="bottom",
            )
        return True

    gates = spec.get(
        "gates",
        [],
    )
    if not isinstance(
        gates,
        list,
    ):
        return False

    gate_dicts = [
        gate
        for gate in gates
        if isinstance(
            gate,
            dict,
        )
    ]

    if (
        len(gate_dicts) != len(labels)
        or len(gate_dicts) != len(gates)
    ):
        return False

    gate_by_id: dict[str, dict[str, Any]] = {}
    consumers: dict[str, list[str]] = {}

    for gate in gate_dicts:
        gate_id = str(
            gate.get(
                "id",
                "",
            )
            or ""
        ).strip()
        if gate_id:
            gate_by_id[gate_id] = gate

    for gate in gate_dicts:
        consumer_id = str(
            gate.get(
                "id",
                "",
            )
            or ""
        ).strip()
        for ref in gate.get(
            "inputs",
            [],
        ):
            ref_id = str(ref).strip()
            if ref_id in gate_by_id:
                consumers.setdefault(
                    ref_id,
                    [],
                ).append(
                    consumer_id
                )

    label_by_gate_id = {
        str(gate.get("id", "") or "").strip(): label
        for gate, label in zip(
            gate_dicts,
            labels,
        )
    }

    compressed_groups: list[dict[str, Any]] = []
    consumed_ids: set[str] = set()
    base_to_compound = {
        "AND": "nand",
        "OR": "nor",
        "XOR": "xnor",
    }

    for gate in gate_dicts:
        gate_id = str(
            gate.get(
                "id",
                "",
            )
            or ""
        ).strip()
        if not gate_id or gate_id in consumed_ids:
            continue

        gate_type = str(
            gate.get(
                "type",
                "",
            )
            or ""
        ).strip().upper()

        consumer_ids = consumers.get(
            gate_id,
            [],
        )

        # logicparse folds BASE -> NOT into N(BASE). Preserve both labels by
        # associating them with the single compound rendered element.
        if (
            gate_type in base_to_compound
            and len(consumer_ids) == 1
        ):
            not_gate = gate_by_id.get(
                consumer_ids[0]
            )
            if isinstance(
                not_gate,
                dict,
            ):
                not_gate_id = str(
                    not_gate.get(
                        "id",
                        "",
                    )
                    or ""
                ).strip()
                not_type = str(
                    not_gate.get(
                        "type",
                        "",
                    )
                    or ""
                ).strip().upper()
                not_inputs = [
                    str(value).strip()
                    for value in not_gate.get(
                        "inputs",
                        [],
                    )
                ]

                if (
                    not_type == "NOT"
                    and not_inputs == [gate_id]
                ):
                    compressed_groups.append(
                        {
                            "render_type": base_to_compound[gate_type],
                            "labels": [
                                label_by_gate_id.get(
                                    gate_id,
                                    "",
                                ),
                                label_by_gate_id.get(
                                    not_gate_id,
                                    "",
                                ),
                            ],
                        }
                    )
                    consumed_ids.add(
                        gate_id
                    )
                    consumed_ids.add(
                        not_gate_id
                    )
                    continue

        compressed_groups.append(
            {
                "render_type": gate_type.casefold(),
                "labels": [
                    label_by_gate_id.get(
                        gate_id,
                        "",
                    )
                ],
            }
        )
        consumed_ids.add(
            gate_id
        )

    if len(compressed_groups) != len(gate_elements):
        return False

    # Keep the mapping conservative: only apply the grouped fallback when the
    # expected and rendered gate kinds agree in order.
    for group, (_, rendered_type) in zip(
        compressed_groups,
        gate_elements,
    ):
        if str(
            group.get(
                "render_type",
                "",
            )
        ).casefold() != rendered_type:
            return False

    for group, (element, _) in zip(
        compressed_groups,
        gate_elements,
    ):
        group_labels = [
            str(value or "").strip()
            for value in group.get(
                "labels",
                [],
            )
            if str(value or "").strip()
        ]

        if not group_labels:
            continue

        add_label(
            element,
            group_labels[0],
            loc="bottom",
        )

        if len(group_labels) > 1:
            # The second identifier belongs to the explicit NOT stage that
            # SchemDraw has represented as the compound gate's output bubble.
            add_label(
                element,
                group_labels[1],
                loc="right",
            )

    return True


def _question_explicitly_requests_truth_table_completion(
    question_text_value: Any,
) -> bool:
    """
    Return True only when complete/fill is grammatically attached to the table
    itself. This prevents wording such as "complete the circuit ... truth
    table given below" from creating an unrelated blank response table.
    """
    text_value = str(
        question_text_value or ""
    )

    direct_patterns = [
        (
            r"\b(?:complete|fill(?:\s+in)?|finish|draw|construct|create|produce)\s+"
            r"(?:the\s+|this\s+|a\s+|following\s+|given\s+|"
            r"shown\s+|provided\s+)?"
            r"(?:truth\s+)?table\b"
        ),
        (
            r"\b(?:truth\s+)?table\b"
            r"[^.!?\n]{0,45}"
            r"\b(?:is\s+|are\s+|should\s+be\s+|must\s+be\s+|to\s+be\s+)?"
            r"(?:complete(?:d)?|fill(?:ed)?(?:\s+in)?|finish(?:ed)?|drawn|constructed|created|produced)\b"
        ),
    ]

    return any(
        re.search(
            pattern,
            text_value,
            flags=re.IGNORECASE,
        )
        is not None
        for pattern in direct_patterns
    )


def _logic_response_scaffold(
    question: dict[str, Any],
) -> dict[str, Any] | None:
    scaffold = question.get(
        "learner_response_scaffold",
        question.get("response_scaffold", {}),
    )

    if not isinstance(
        scaffold,
        dict,
    ):
        return None

    if str(
        scaffold.get(
            "type",
            "",
        )
        or ""
    ).strip().casefold() != "truth_table":
        return None

    # Notebook 06 can carry a deterministic scaffold from earlier wording.
    # Render it only when the final learner-facing question still explicitly
    # asks for the table itself to be completed/filled.
    if not _question_explicitly_requests_truth_table_completion(
        question.get(
            "question_text",
            "",
        )
    ):
        return None

    columns = scaffold.get(
        "columns",
        [],
    )
    rows = scaffold.get(
        "rows",
        [],
    )

    if (
        not isinstance(
            columns,
            list,
        )
        or not columns
        or not isinstance(
            rows,
            list,
        )
        or not rows
    ):
        return None

    return scaffold


def _combine_logic_and_truth_table(
    *,
    logic_path: Path,
    output_path: Path,
    scaffold: dict[str, Any],
) -> None:
    """
    Keep the SchemDraw circuit as the stimulus and append the separate blank
    learner truth-table response area underneath it in the same rendered asset.
    """
    logic_image = plt.imread(
        logic_path
    )

    rows = scaffold.get(
        "rows",
        [],
    )

    columns = [
        str(value)
        for value in scaffold.get(
            "columns",
            [],
        )
    ]

    display_rows = [
        [
            ""
            if value is None
            else str(value)
            for value in row
        ]
        for row in rows
        if isinstance(
            row,
            list,
        )
    ]

    if not display_rows:
        logic_path.replace(
            output_path
        )
        return

    image_height = max(
        2.4,
        7.2
        * (
            logic_image.shape[0]
            / max(
                1,
                logic_image.shape[1],
            )
        ),
    )

    table_height = max(
        1.8,
        0.48
        * (
            len(display_rows)
            + 1
        ),
    )

    fig = plt.figure(
        figsize=(
            9.0,
            image_height
            + table_height,
        )
    )

    grid = fig.add_gridspec(
        2,
        1,
        height_ratios=[
            image_height,
            table_height,
        ],
        hspace=0.04,
    )

    logic_ax = fig.add_subplot(
        grid[0]
    )

    logic_ax.imshow(
        logic_image
    )

    logic_ax.axis(
        "off"
    )

    table_ax = fig.add_subplot(
        grid[1]
    )

    table_ax.axis(
        "off"
    )

    table = table_ax.table(
        cellText=display_rows,
        colLabels=columns,
        cellLoc="center",
        loc="center",
    )

    table.auto_set_font_size(
        False
    )

    table.set_fontsize(
        10
    )

    table.scale(
        1.0,
        1.45,
    )

    for cell in table.get_celld().values():
        cell.set_edgecolor(
            "black"
        )

        cell.set_linewidth(
            1.0
        )

        cell.set_facecolor(
            "white"
        )

        cell.get_text().set_color(
            "black"
        )

    save_matplotlib_figure(
        fig,
        output_path,
    )

    try:
        logic_path.unlink()
    except OSError:
        pass


def _logic_gate_is_placeholder(
    gate: dict[str, Any],
) -> bool:
    render_mode = str(
        gate.get(
            "render_mode",
            "",
        )
        or ""
    ).strip().casefold()

    if render_mode in {
        "labelled_placeholder",
        "placeholder",
        "hidden_type",
    }:
        return True

    if bool(gate.get("hide_type")):
        return True

    student_visible_type = str(
        gate.get(
            "student_visible_type",
            "",
        )
        or ""
    ).strip().casefold()

    return student_visible_type == "placeholder"


def _logic_question_uses_placeholder_mode(
    question: dict[str, Any],
    spec: dict[str, Any],
) -> bool:
    if str(
        question.get(
            "logic_gate_display_mode",
            "",
        )
        or ""
    ).strip().casefold() in {
        "labelled_placeholder",
        "placeholder",
    }:
        return True

    for gate in spec.get(
        "gates",
        [],
    ):
        if isinstance(gate, dict) and _logic_gate_is_placeholder(gate):
            return True

    return False


def _render_logic_placeholder_diagram(
    question: dict[str, Any],
    spec: dict[str, Any],
    path: Path,
) -> None:
    from matplotlib import pyplot as plt
    from matplotlib.patches import Rectangle

    inputs = [
        str(value).strip()
        for value in spec.get("inputs", [])
        if str(value).strip()
    ]

    gates = [
        gate
        for gate in spec.get("gates", [])
        if isinstance(gate, dict)
    ]

    if not inputs or not gates:
        raise ValueError(
            "Placeholder logic rendering requires non-empty inputs and gates."
        )

    gate_by_id = {}
    for gate in gates:
        gate_id = str(gate.get("id", "") or "").strip()
        if gate_id:
            gate_by_id[gate_id] = gate

    memo_depth: dict[str, int] = {}

    def ref_depth(ref: str) -> int:
        if ref in inputs:
            return 0
        if ref in memo_depth:
            return memo_depth[ref]
        gate = gate_by_id.get(ref)
        if gate is None:
            memo_depth[ref] = 0
            return 0
        child_depths = [
            ref_depth(str(child).strip())
            for child in gate.get("inputs", [])
        ]
        depth = 1 + (max(child_depths) if child_depths else 0)
        memo_depth[ref] = depth
        return depth

    layer_map: dict[int, list[dict[str, Any]]] = {}
    max_depth = 1
    for gate in gates:
        gate_id = str(gate.get("id", "") or "").strip()
        depth = ref_depth(gate_id) if gate_id else 1
        layer_map.setdefault(depth, []).append(gate)
        max_depth = max(max_depth, depth)

    input_y = {}
    if len(inputs) == 1:
        input_positions = [0.5]
    else:
        input_positions = [
            0.90 - index * (0.75 / (len(inputs) - 1))
            for index in range(len(inputs))
        ]
    for name, y in zip(inputs, input_positions):
        input_y[name] = y

    node_pos: dict[str, tuple[float, float]] = {}
    x_by_depth = {
        depth: 0.28 + (depth - 1) * (0.48 / max(1, max_depth - 1))
        for depth in range(1, max_depth + 1)
    }

    for depth in sorted(layer_map):
        layer = layer_map[depth]
        estimates = []
        for order, gate in enumerate(layer):
            refs = [str(value).strip() for value in gate.get("inputs", [])]
            source_ys = []
            for ref in refs:
                if ref in input_y:
                    source_ys.append(input_y[ref])
                elif ref in node_pos:
                    source_ys.append(node_pos[ref][1])
            estimate = sum(source_ys) / len(source_ys) if source_ys else 0.5
            estimates.append((estimate, order, gate))

        estimates.sort(key=lambda item: (-item[0], item[1]))
        count = len(estimates)
        if count == 1:
            layer_ys = [0.5]
        else:
            layer_ys = [
                0.90 - index * (0.75 / (count - 1))
                for index in range(count)
            ]

        for y, (_, _, gate) in zip(layer_ys, estimates):
            gate_id = str(gate.get("id", "") or "").strip()
            if gate_id:
                node_pos[gate_id] = (x_by_depth[depth], y)

    consumers: dict[str, list[str]] = {}
    for gate in gates:
        gate_id = str(gate.get("id", "") or "").strip()
        for ref in gate.get("inputs", []):
            ref_id = str(ref).strip()
            if ref_id in gate_by_id:
                consumers.setdefault(ref_id, []).append(gate_id)

    fig_width = max(7.0, 4.8 + 0.8 * max_depth)
    fig_height = max(3.4, 2.0 + 0.7 * max(len(gates), len(inputs)))
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    box_w = 0.11
    box_h = 0.08
    input_x = 0.08
    output_x = 0.92

    for name, y in input_y.items():
        ax.text(input_x - 0.02, y, name, ha="right", va="center", fontsize=12)
        ax.plot([input_x, input_x + 0.05], [y, y], color="black", linewidth=1.6)

    def source_anchor(ref: str) -> tuple[float, float]:
        if ref in input_y:
            return input_x + 0.05, input_y[ref]
        if ref in node_pos:
            x, y = node_pos[ref]
            return x + box_w / 2, y
        return input_x + 0.05, 0.5

    # Draw connections first.
    for gate in gates:
        gate_id = str(gate.get("id", "") or "").strip()
        if gate_id not in node_pos:
            continue
        x, y = node_pos[gate_id]
        refs = [str(value).strip() for value in gate.get("inputs", [])]
        port_count = max(1, len(refs))
        if port_count == 1:
            target_ys = [y]
        elif port_count == 2:
            target_ys = [y + 0.018, y - 0.018]
        else:
            step = 0.05 / max(1, port_count - 1)
            start = y + 0.025
            target_ys = [start - idx * step for idx in range(port_count)]

        target_x = x - box_w / 2

        # Give every incoming reference its own routing column. Using one shared
        # mid_x for two gate inputs visually/electrically joined independent
        # signals (for example A and B) before they reached the placeholder.
        # Staggered columns preserve separate physical input ports while still
        # allowing the same source signal to branch legitimately to multiple
        # downstream gates.
        route_spacing = 0.026
        nearest_route_x = target_x - 0.020

        for port_index, (ref, target_y) in enumerate(zip(refs, target_ys)):
            sx, sy = source_anchor(ref)
            mid_x = nearest_route_x - (port_index * route_spacing)

            ax.plot([sx, mid_x], [sy, sy], color="black", linewidth=1.3)
            ax.plot([mid_x, mid_x], [sy, target_y], color="black", linewidth=1.3)
            ax.plot([mid_x, target_x], [target_y, target_y], color="black", linewidth=1.3)

    output = spec.get("output", {})
    if not isinstance(output, dict):
        output = {}
    output_ref = str(output.get("from", "") or "").strip()
    output_label = str(output.get("label", "Q") or "Q").strip() or "Q"

    for gate in gates:
        gate_id = str(gate.get("id", "") or "").strip()
        if gate_id not in node_pos:
            continue
        x, y = node_pos[gate_id]
        label = str(gate.get("label", "") or gate.get("student_visible_label", "") or "").strip()
        is_placeholder = _logic_gate_is_placeholder(gate)
        gate_type = str(gate.get("type", "") or "").strip().upper()

        rect = Rectangle(
            (x - box_w / 2, y - box_h / 2),
            box_w,
            box_h,
            linewidth=1.6,
            edgecolor="black",
            facecolor="white",
        )
        ax.add_patch(rect)

        visible_text = label or gate_id
        if not is_placeholder:
            visible_text = gate_type if gate_type else visible_text
            if label and gate_type:
                ax.text(x, y - 0.075, label, ha="center", va="top", fontsize=9)
        ax.text(x, y, visible_text, ha="center", va="center", fontsize=11)

        is_sink = gate_id == output_ref or gate_id not in consumers
        if is_sink:
            right_x = x + box_w / 2
            end_x = output_x if gate_id == output_ref else min(output_x - 0.06, right_x + 0.08)
            ax.plot([right_x, end_x], [y, y], color="black", linewidth=1.5)
            if gate_id == output_ref:
                ax.text(output_x + 0.015, y, output_label, ha="left", va="center", fontsize=12)

    fig.savefig(path, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)

def render_logic_visual(
    question: dict[str, Any],
) -> Path:
    spec = question[
        "visual"
    ][
        "spec"
    ]

    path = asset_path(
        question,
        "logic_gate_diagram",
        "png",
    )

    scaffold = (
        _logic_response_scaffold(
            question
        )
    )

    use_placeholder_mode = _logic_question_uses_placeholder_mode(
        question,
        spec,
    )

    if use_placeholder_mode:
        if scaffold is None:
            _render_logic_placeholder_diagram(
                question,
                spec,
                path,
            )
            return path

        base_path = path.with_name(
            path.stem
            + "_circuit_only"
            + path.suffix
        )

        _render_logic_placeholder_diagram(
            question,
            spec,
            base_path,
        )

        _combine_logic_and_truth_table(
            logic_path=base_path,
            output_path=path,
            scaffold=scaffold,
        )

        return path

    expression, output_label = (
        _logic_expression_from_spec(
            spec
        )
    )

    drawing = logicparse(
        expression,
        outlabel=output_label,
    )

    gate_labels = (
        _logic_gate_labels_for_render(
            question,
            spec,
        )
    )

    _attach_logic_gate_position_labels(
        drawing,
        gate_labels,
        spec,
    )

    if scaffold is None:
        drawing.save(
            str(path)
        )

        return path

    base_path = path.with_name(
        path.stem
        + "_circuit_only"
        + path.suffix
    )

    drawing.save(
        str(base_path)
    )

    _combine_logic_and_truth_table(
        logic_path=base_path,
        output_path=path,
        scaffold=scaffold,
    )

    return path


# Backward-compatible alias while MCP is not yet wired.
render_logic_diagram = render_logic_visual


## 5. Kroki renderer — technical diagrams

Kroki is the technical-diagram backend.

The current Notebook 06 visual contracts are translated into GraphViz DOT
source and sent to Kroki:

- `network_diagram`
- `simple_flowchart`
- `cpu_block_diagram`

This gives automatic layout without maintaining local hand-positioned diagram
code. The renderer boundary is intentionally generic so future Kroki engines
(Mermaid, WaveDrom, Bytefield, ERD, etc.) can be added without changing the
quiz-generation LLM contract.


In [ ]:
def _dot_escape(value: Any) -> str:
    return (
        str(value or "")
        .replace("\\", "\\\\")
        .replace('"', '\\"')
        .replace("\n", "\\n")
    )


def _dot_id(value: Any) -> str:
    return '"' + _dot_escape(value) + '"'


def _build_network_dot(
    spec: dict[str, Any],
) -> str:
    shape_map = {
        "router": "diamond",
        "switch": "box3d",
        "server": "cylinder",
        "computer": "box",
        "pc": "box",
        "laptop": "box",
        "printer": "box",
        "access_point": "ellipse",
        "wireless_access_point": "ellipse",
    }

    lines = [
        "digraph G {",
        '  graph [rankdir=LR, bgcolor="white", pad="0.15", '
        'nodesep="0.5", ranksep="0.7"];',
        '  node [shape=box, style="rounded", color="black", '
        'fontname="Arial"];',
        '  edge [color="black", fontname="Arial"];',
    ]

    for node in spec.get("nodes", []):
        if not isinstance(node, dict):
            continue

        node_id = str(node.get("id", "") or "").strip()
        if not node_id:
            continue

        node_type = str(
            node.get("type", "") or ""
        ).strip().casefold()

        label = str(
            node.get("label", node_id)
            or node_id
        )

        shape = shape_map.get(
            node_type,
            "box",
        )

        lines.append(
            f"  {_dot_id(node_id)} "
            f'[label="{_dot_escape(label)}", shape="{shape}"];'
        )

    for edge in spec.get("edges", []):
        if not isinstance(edge, dict):
            continue

        source = str(edge.get("from", "") or "").strip()
        target = str(edge.get("to", "") or "").strip()

        if not source or not target:
            continue

        label = str(edge.get("label", "") or "")

        lines.append(
            f"  {_dot_id(source)} -> {_dot_id(target)} "
            f'[label="{_dot_escape(label)}"];'
        )

    lines.append("}")
    return "\n".join(lines)


def _build_flowchart_dot(
    spec: dict[str, Any],
) -> str:
    shape_map = {
        "start_end": "oval",
        "start": "oval",
        "end": "oval",
        "process": "box",
        "decision": "diamond",
        "input_output": "parallelogram",
        "io": "parallelogram",
    }

    lines = [
        "digraph G {",
        '  graph [rankdir=TB, bgcolor="white", pad="0.15", '
        'nodesep="0.4", ranksep="0.55"];',
        '  node [color="black", fontname="Arial"];',
        '  edge [color="black", fontname="Arial"];',
    ]

    for node in spec.get("nodes", []):
        if not isinstance(node, dict):
            continue

        node_id = str(node.get("id", "") or "").strip()
        if not node_id:
            continue

        node_type = str(
            node.get("type", "process")
            or "process"
        ).strip().casefold()

        text = str(
            node.get(
                "text",
                node.get("label", node_id),
            )
            or node_id
        )

        shape = shape_map.get(
            node_type,
            "box",
        )

        lines.append(
            f"  {_dot_id(node_id)} "
            f'[label="{_dot_escape(text)}", shape="{shape}"];'
        )

    for edge in spec.get("edges", []):
        if not isinstance(edge, dict):
            continue

        source = str(edge.get("from", "") or "").strip()
        target = str(edge.get("to", "") or "").strip()

        if not source or not target:
            continue

        label = str(edge.get("label", "") or "")

        lines.append(
            f"  {_dot_id(source)} -> {_dot_id(target)} "
            f'[label="{_dot_escape(label)}"];'
        )

    lines.append("}")
    return "\n".join(lines)


def _build_cpu_dot(
    spec: dict[str, Any],
) -> str:
    lines = [
        "digraph G {",
        '  graph [rankdir=LR, bgcolor="white", pad="0.15", '
        'nodesep="0.55", ranksep="0.75"];',
        '  node [shape=box, style="rounded", color="black", '
        'fontname="Arial"];',
        '  edge [color="black", fontname="Arial"];',
    ]

    for component in spec.get("components", []):
        if not isinstance(component, dict):
            continue

        component_id = str(
            component.get("id", "") or ""
        ).strip()

        if not component_id:
            continue

        label = str(
            component.get(
                "label",
                component_id,
            )
            or component_id
        )

        lines.append(
            f"  {_dot_id(component_id)} "
            f'[label="{_dot_escape(label)}"];'
        )

    for connection in spec.get("connections", []):
        if not isinstance(connection, dict):
            continue

        source = str(
            connection.get("from", "") or ""
        ).strip()

        target = str(
            connection.get("to", "") or ""
        ).strip()

        if not source or not target:
            continue

        label = str(
            connection.get("label", "") or ""
        )

        lines.append(
            f"  {_dot_id(source)} -> {_dot_id(target)} "
            f'[label="{_dot_escape(label)}"];'
        )

    lines.append("}")
    return "\n".join(lines)


def _kroki_post(
    *,
    diagram_type: str,
    source: str,
    output_format: str = KROKI_OUTPUT_FORMAT,
) -> bytes:
    url = (
        f"{KROKI_ENDPOINT}/"
        f"{diagram_type}/"
        f"{output_format}"
    )

    request = urllib.request.Request(
        url=url,
        data=source.encode("utf-8"),
        method="POST",
        headers={
            "Content-Type": "text/plain; charset=utf-8",
            "Accept": (
                "image/png"
                if output_format == "png"
                else "image/svg+xml"
            ),
            "User-Agent": "EDTech-Agent2-Notebook08/2.0",
        },
    )

    try:
        with urllib.request.urlopen(
            request,
            timeout=KROKI_TIMEOUT_SECONDS,
        ) as response:
            payload = response.read()

            if not payload:
                raise RuntimeError(
                    "Kroki returned an empty response."
                )

            return payload

    except urllib.error.HTTPError as exc:
        body = ""
        try:
            body = exc.read().decode(
                "utf-8",
                errors="replace",
            )
        except Exception:
            pass

        raise RuntimeError(
            "Kroki HTTP error "
            f"{exc.code}: {body[:500]}"
        ) from exc

    except urllib.error.URLError as exc:
        raise RuntimeError(
            f"Kroki connection error: {exc.reason}"
        ) from exc


def render_technical_visual(
    question: dict[str, Any],
) -> tuple[Path, str]:
    visual_type = normalize_visual_type(
        question.get(
            "visual_requirement",
            "none",
        )
    )

    spec = question["visual"]["spec"]

    if visual_type == "network_diagram":
        source = _build_network_dot(spec)

    elif visual_type == "simple_flowchart":
        source = _build_flowchart_dot(spec)

    elif visual_type == "cpu_block_diagram":
        source = _build_cpu_dot(spec)

    else:
        raise ValueError(
            "Kroki technical renderer does not support "
            f"{visual_type}."
        )

    # Current technical contracts all use GraphViz through Kroki.
    engine = "graphviz"

    path = asset_path(
        question,
        visual_type,
        KROKI_OUTPUT_FORMAT,
    )

    payload = _kroki_post(
        diagram_type=engine,
        source=source,
        output_format=KROKI_OUTPUT_FORMAT,
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    path.write_bytes(payload)

    return (
        path,
        f"kroki_{engine}",
    )


# Compatibility alias for any temporary caller using the previous name.
render_graph_diagram = render_technical_visual


## 6. Local structured renderer — exact-value assessment visuals

The local renderer is intentionally **narrowed**, not removed.

It is used only for structured visuals where deterministic cells/values are more
important than graph layout:

- code blocks;
- trace tables;
- truth tables;
- arrays;
- database tables;
- memory grids;
- binary registers.

It does **not** position network nodes, flowchart shapes or CPU components.
Those technical diagrams now go to Kroki.


In [ ]:
def _render_table_png(
    *,
    question: dict[str, Any],
    visual_type: str,
    columns: list[Any],
    rows: list[Any],
    caption: str = "",
    fontsize: float = 10.0,
) -> Path:
    normalized_columns = [
        str(
            ""
            if value is None
            else value
        )
        for value in columns
    ]

    normalized_rows: list[list[Any]] = []

    for row in rows:
        values = (
            list(row)
            if isinstance(
                row,
                (list, tuple),
            )
            else [row]
        )

        padded = (
            values
            + [""]
            * max(
                0,
                len(normalized_columns)
                - len(values),
            )
        )[
            : len(normalized_columns)
        ]

        normalized_rows.append(
            [
                ""
                if value is None
                else value
                for value in padded
            ]
        )

    width = max(
        5.0,
        min(
            13.0,
            1.25
            * max(
                2,
                len(
                    normalized_columns
                ),
            ),
        ),
    )

    height = max(
        2.0,
        min(
            10.0,
            1.2
            + 0.46
            * (
                len(
                    normalized_rows
                )
                + 1
            ),
        ),
    )

    fig, ax = plt.subplots(
        figsize=(
            width,
            height,
        )
    )

    ax.axis(
        "off"
    )

    table = ax.table(
        cellText=normalized_rows,
        colLabels=normalized_columns,
        loc="center",
        cellLoc="center",
    )

    table.auto_set_font_size(
        False
    )

    table.set_fontsize(
        fontsize
    )

    table.scale(
        1.0,
        1.45,
    )

    if caption:
        ax.set_title(
            caption,
            fontsize=11,
            pad=10,
        )

    path = asset_path(
        question,
        visual_type,
        "png",
    )

    save_matplotlib_figure(
        fig,
        path,
    )

    return path


def _render_code_block_png(
    question: dict[str, Any],
) -> Path:
    spec = question[
        "visual"
    ][
        "spec"
    ]

    code = str(
        spec.get(
            "code",
            "",
        )
        or ""
    ).rstrip()

    caption = str(
        spec.get(
            "caption",
            "",
        )
        or ""
    ).strip()

    lines = (
        code.splitlines()
        or [""]
    )

    longest = max(
        len(line)
        for line in lines
    )

    width = min(
        12.0,
        max(
            5.0,
            1.2
            + 0.085
            * longest,
        ),
    )

    height = min(
        8.0,
        max(
            1.8,
            0.8
            + 0.32
            * len(lines),
        ),
    )

    fig, ax = plt.subplots(
        figsize=(
            width,
            height,
        )
    )

    ax.set_xlim(
        0,
        1,
    )

    ax.set_ylim(
        0,
        1,
    )

    ax.axis(
        "off"
    )

    ax.add_patch(
        Rectangle(
            (
                0.02,
                0.06,
            ),
            0.96,
            0.88,
            edgecolor="black",
            facecolor="white",
            linewidth=1.1,
        )
    )

    ax.text(
        0.05,
        0.88,
        code,
        ha="left",
        va="top",
        family="monospace",
        fontsize=10,
    )

    if caption:
        ax.set_title(
            caption,
            fontsize=11,
            pad=8,
        )

    path = asset_path(
        question,
        "code_block",
        "png",
    )

    save_matplotlib_figure(
        fig,
        path,
    )

    return path


def render_structured_visual(
    question: dict[str, Any],
) -> Path:
    visual_type = normalize_visual_type(
        question.get(
            "visual_requirement",
            "none",
        )
    )

    spec = question[
        "visual"
    ][
        "spec"
    ]

    caption = str(
        spec.get(
            "caption",
            "",
        )
        or ""
    ).strip()

    if visual_type == "code_block":
        return _render_code_block_png(
            question
        )

    if visual_type == "truth_table":
        return _render_table_png(
            question=question,
            visual_type=visual_type,
            columns=spec.get(
                "columns",
                [],
            ),
            rows=spec.get(
                "rows",
                [],
            ),
            caption=caption,
        )

    if visual_type == "trace_table":
        return _render_table_png(
            question=question,
            visual_type=visual_type,
            columns=spec.get(
                "columns",
                [],
            ),
            rows=spec.get(
                "rows",
                [],
            ),
            caption=caption,
        )

    if visual_type == "database_table":
        columns = [
            str(value)
            for value in spec.get(
                "columns",
                [],
            )
        ]

        primary_key = str(
            spec.get(
                "primary_key",
                "",
            )
            or ""
        ).strip()

        display_columns = [
            (
                f"{column} (PK)"
                if column
                == primary_key
                else column
            )
            for column in columns
        ]

        return _render_table_png(
            question=question,
            visual_type=visual_type,
            columns=display_columns,
            rows=spec.get(
                "rows",
                [],
            ),
            caption=caption,
        )

    if visual_type == "array_grid":
        values = spec.get(
            "values",
            [],
        )

        if values and not isinstance(
            values[0],
            list,
        ):
            values = [
                values
            ]

        max_cols = max(
            (
                len(row)
                for row in values
                if isinstance(
                    row,
                    list,
                )
            ),
            default=0,
        )

        column_labels = spec.get(
            "column_labels",
            [],
        )

        if (
            not isinstance(
                column_labels,
                list,
            )
            or len(column_labels)
            != max_cols
        ):
            column_labels = [
                str(index)
                for index in range(
                    max_cols
                )
            ]

        row_labels = spec.get(
            "row_labels",
            [],
        )

        normalized_rows: list[list[Any]] = []

        for index, row in enumerate(
            values
        ):
            row_values = (
                list(row)
                if isinstance(
                    row,
                    list,
                )
                else [row]
            )

            if (
                isinstance(
                    row_labels,
                    list,
                )
                and len(
                    row_labels
                )
                == len(
                    values
                )
            ):
                normalized_rows.append(
                    [
                        row_labels[
                            index
                        ]
                    ]
                    + row_values
                )
            else:
                normalized_rows.append(
                    row_values
                )

        columns = (
            [
                ""
            ]
            + column_labels
            if (
                isinstance(
                    row_labels,
                    list,
                )
                and len(
                    row_labels
                )
                == len(
                    values
                )
            )
            else column_labels
        )

        return _render_table_png(
            question=question,
            visual_type=visual_type,
            columns=columns,
            rows=normalized_rows,
            caption=caption,
        )

    if visual_type == "memory_grid":
        addresses = spec.get(
            "addresses",
            [],
        )

        values = spec.get(
            "values",
            [],
        )

        rows = [
            [
                address,
                values[index]
                if index
                < len(values)
                else "",
            ]
            for index, address
            in enumerate(
                addresses
            )
        ]

        return _render_table_png(
            question=question,
            visual_type=visual_type,
            columns=[
                "Address",
                "Value",
            ],
            rows=rows,
            caption=caption,
        )

    if visual_type == "binary_register":
        bits = spec.get(
            "bits",
            [],
        )

        place_values = spec.get(
            "place_values",
            [],
        )

        labels = spec.get(
            "labels",
            [],
        )

        if (
            isinstance(
                labels,
                list,
            )
            and len(labels)
            == len(bits)
        ):
            columns = labels

        elif (
            isinstance(
                place_values,
                list,
            )
            and len(
                place_values
            )
            == len(bits)
        ):
            columns = place_values

        else:
            columns = list(
                range(
                    len(bits) - 1,
                    -1,
                    -1,
                )
            )

        return _render_table_png(
            question=question,
            visual_type=visual_type,
            columns=columns,
            rows=[
                bits
            ],
            caption=caption,
            fontsize=11,
        )

    raise ValueError(
        f"Structured renderer does not support {visual_type}."
    )


## 7. Three-backend renderer registry and temporary dispatcher

This registry is the stable tool boundary.

Today:

```text
Notebook 08 local dispatcher
   ├─ render_logic_visual()      → SchemDraw
   ├─ render_technical_visual()  → Kroki
   └─ render_structured_visual() → local deterministic renderer
```

Later:

```text
LLM / controller
      ↓
MCP tool selection
      ↓
same three contracts
```

The LLM is not allowed to choose arbitrary rendering libraries directly. MCP
will later select only from these approved tool contracts.


In [ ]:
RENDERER_REGISTRY: dict[
    str,
    dict[str, Any],
] = {
    "logic_gate_diagram": {
        "tool_name": "render_logic_visual",
        "backend": "schemdraw",
        "engine": "schemdraw_logic",
        "renderer": "schemdraw",
        "function": render_logic_visual,
    },

    "network_diagram": {
        "tool_name": "render_technical_visual",
        "backend": "kroki",
        "engine": "graphviz",
        "renderer": "kroki_graphviz",
        "function": render_technical_visual,
    },
    "simple_flowchart": {
        "tool_name": "render_technical_visual",
        "backend": "kroki",
        "engine": "graphviz",
        "renderer": "kroki_graphviz",
        "function": render_technical_visual,
    },
    "cpu_block_diagram": {
        "tool_name": "render_technical_visual",
        "backend": "kroki",
        "engine": "graphviz",
        "renderer": "kroki_graphviz",
        "function": render_technical_visual,
    },

    "truth_table": {
        "tool_name": "render_structured_visual",
        "backend": "local_structured",
        "engine": "table",
        "renderer": "local_structured",
        "function": render_structured_visual,
    },
    "code_block": {
        "tool_name": "render_structured_visual",
        "backend": "local_structured",
        "engine": "code",
        "renderer": "local_structured",
        "function": render_structured_visual,
    },
    "trace_table": {
        "tool_name": "render_structured_visual",
        "backend": "local_structured",
        "engine": "table",
        "renderer": "local_structured",
        "function": render_structured_visual,
    },
    "array_grid": {
        "tool_name": "render_structured_visual",
        "backend": "local_structured",
        "engine": "grid",
        "renderer": "local_structured",
        "function": render_structured_visual,
    },
    "database_table": {
        "tool_name": "render_structured_visual",
        "backend": "local_structured",
        "engine": "table",
        "renderer": "local_structured",
        "function": render_structured_visual,
    },
    "memory_grid": {
        "tool_name": "render_structured_visual",
        "backend": "local_structured",
        "engine": "grid",
        "renderer": "local_structured",
        "function": render_structured_visual,
    },
    "binary_register": {
        "tool_name": "render_structured_visual",
        "backend": "local_structured",
        "engine": "grid",
        "renderer": "local_structured",
        "function": render_structured_visual,
    },
}


def render_assessment_diagram(
    question: dict[str, Any],
) -> dict[str, Any]:
    visual_type = normalize_visual_type(
        question.get(
            "visual_requirement",
            "none",
        )
    )

    if visual_type == "none":
        return {
            "status": "not_required",
            "visual_type": "none",
            "tool_name": None,
            "backend": None,
            "engine": None,
            "renderer": None,
            "path": None,
            "sha256": None,
            "errors": [],
        }

    validation_errors = (
        validate_visual_question(
            question
        )
    )

    if validation_errors:
        return {
            "status": "invalid_spec",
            "visual_type": visual_type,
            "tool_name": None,
            "backend": None,
            "engine": None,
            "renderer": None,
            "path": None,
            "sha256": None,
            "errors": validation_errors,
        }

    route = RENDERER_REGISTRY.get(
        visual_type
    )

    if route is None:
        return {
            "status": "unsupported",
            "visual_type": visual_type,
            "tool_name": None,
            "backend": None,
            "engine": None,
            "renderer": None,
            "path": None,
            "sha256": None,
            "errors": [
                f"No renderer route registered for {visual_type}."
            ],
        }

    renderer_name = str(
        route.get(
            "renderer",
            route.get("backend", ""),
        )
        or ""
    )

    try:
        result = route[
            "function"
        ](
            question
        )

        if (
            isinstance(
                result,
                tuple,
            )
            and len(result)
            == 2
        ):
            path, actual_renderer = result
            renderer_name = str(
                actual_renderer
            )
        else:
            path = result

        path = Path(
            path
        )

        if (
            not path.is_file()
            or path.stat().st_size
            <= 0
        ):
            raise RuntimeError(
                "Renderer did not produce a non-empty asset."
            )

        return {
            "status": "rendered",
            "visual_type": visual_type,
            "tool_name": route[
                "tool_name"
            ],
            "backend": route[
                "backend"
            ],
            "engine": route[
                "engine"
            ],
            "renderer": renderer_name,
            "path": str(path),
            "sha256": file_sha256(
                path
            ),
            "errors": [],
        }

    except Exception as exc:
        return {
            "status": "render_failed",
            "visual_type": visual_type,
            "tool_name": route[
                "tool_name"
            ],
            "backend": route[
                "backend"
            ],
            "engine": route[
                "engine"
            ],
            "renderer": renderer_name,
            "path": None,
            "sha256": None,
            "errors": [
                f"{type(exc).__name__}: {exc}"
            ],
        }


## 8. Batch render the Notebook 06 handoff

This is the temporary pre-MCP execution path.

Each visual question is routed through the registry, rendered, hashed and written to the Notebook 08 results payload. Text-only questions pass through without a renderer call.


In [ ]:
rendered_questions: list[
    dict[str, Any]
] = []

render_results: list[
    dict[str, Any]
] = []

for question in handoff_questions:
    if not isinstance(
        question,
        dict,
    ):
        continue

    rendered_question = deepcopy(
        question
    )

    result = (
        render_assessment_diagram(
            rendered_question
        )
    )

    rendered_question[
        "notebook08_visual_status"
    ] = result.get(
        "status"
    )

    rendered_question[
        "notebook08_visual_tool_name"
    ] = result.get(
        "tool_name"
    )

    rendered_question[
        "notebook08_visual_renderer"
    ] = result.get(
        "renderer"
    )

    rendered_question[
        "notebook08_visual_backend"
    ] = result.get(
        "backend"
    )

    rendered_question[
        "notebook08_visual_engine"
    ] = result.get(
        "engine"
    )

    rendered_question[
        "notebook08_visual_path"
    ] = result.get(
        "path"
    )

    rendered_question[
        "notebook08_visual_asset_sha256"
    ] = result.get(
        "sha256"
    )

    rendered_question[
        "notebook08_visual_errors"
    ] = result.get(
        "errors",
        [],
    )

    rendered_questions.append(
        rendered_question
    )

    render_results.append(
        {
            "generated_question_id": rendered_question.get(
                "generated_question_id"
            ),
            "question_index": rendered_question.get(
                "question_index"
            ),
            **result,
        }
    )


required_visual_failures = [
    result
    for result in render_results
    if (
        result.get(
            "visual_type"
        )
        != "none"
        and result.get(
            "status"
        )
        != "rendered"
    )
]

visual_integrity_release_eligible = bool(
    not required_visual_failures
)

results_payload = {
    "schema_version": NOTEBOOK08_RESULTS_SCHEMA_VERSION,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source_handoff_path": str(
        HANDOFF_PATH
    ),
    "source_handoff_schema_version": handoff.get(
        "schema_version"
    ),
    "visual_spec_schema_version": handoff.get(
        "visual_spec_schema_version",
        VISUAL_SCHEMA_VERSION,
    ),
    "mcp_status": "NOT_WIRED_YET",
    "separate_image_generation_api_used": False,
    "kroki_endpoint": KROKI_ENDPOINT,
    "kroki_endpoint_is_public": KROKI_ENDPOINT_IS_PUBLIC,
    "kroki_timeout_seconds": KROKI_TIMEOUT_SECONDS,
    "paid_ai_image_generation_used": False,
    "visual_integrity_gate": {
        "status": (
            "PASS"
            if visual_integrity_release_eligible
            else "BLOCKED"
        ),
        "release_eligible": visual_integrity_release_eligible,
        "required_visual_failures": required_visual_failures,
    },
    "renderer_registry": {
        visual_type: {
            "tool_name": route[
                "tool_name"
            ],
            "backend": route[
                "backend"
            ],
            "engine": route[
                "engine"
            ],
            "renderer": route[
                "renderer"
            ],
        }
        for visual_type, route
        in RENDERER_REGISTRY.items()
    },
    "summary": {
        "question_count": len(
            rendered_questions
        ),
        "visual_required_count": sum(
            1
            for result in render_results
            if result.get(
                "visual_type"
            )
            != "none"
        ),
        "rendered_count": sum(
            1
            for result in render_results
            if result.get(
                "status"
            )
            == "rendered"
        ),
        "release_eligible": visual_integrity_release_eligible,
        "fail_closed": bool(
            not visual_integrity_release_eligible
        ),
        "failed_count": sum(
            1
            for result in render_results
            if result.get(
                "status"
            )
            in {
                "invalid_spec",
                "render_failed",
                "unsupported",
            }
        ),
    },
    "render_results": render_results,
    "questions": rendered_questions,
}

RESULTS_PATH.write_text(
    json.dumps(
        results_payload,
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)

print(
    "Saved Notebook 08 results:",
    RESULTS_PATH,
)

print(
    json.dumps(
        results_payload[
            "summary"
        ],
        indent=2,
    )
)


## 9. Patch a downstream manifest copy

For safe migration, Notebook 08 does **not** destructively rewrite the original Notebook 06 manifest.

Instead it creates:

```text
final_quiz_manifest_with_notebook08_visuals.json
```

and replaces matching generated-question visual paths in that copy with the Notebook 08 tool-rendered assets.

This gives Streamlit/PDF integration a clean artifact to test before the MCP controller is wired.


In [ ]:
def _question_match_key(
    question: dict[str, Any],
) -> tuple[str, int]:
    return (
        str(
            question.get(
                "generated_question_id",
                "",
            )
            or ""
        ),
        int(
            question.get(
                "question_index",
                0,
            )
            or 0
        ),
    )


notebook06_manifest_path = (
    NOTEBOOK06_OUTPUT_DIR
    / "final_quiz_manifest.json"
)

if (
    PATCH_NOTEBOOK06_MANIFEST
    and notebook06_manifest_path.is_file()
):
    base_manifest = json.loads(
        notebook06_manifest_path.read_text(
            encoding="utf-8"
        )
    )

    rendered_lookup = {}

    for question in rendered_questions:
        key = _question_match_key(
            question
        )

        rendered_lookup[
            key
        ] = question

    def patch_question_list(
        questions: Any,
    ) -> Any:
        if not isinstance(
            questions,
            list,
        ):
            return questions

        patched = []

        for index, question in enumerate(
            questions,
            start=1,
        ):
            if not isinstance(
                question,
                dict,
            ):
                patched.append(
                    question
                )
                continue

            candidate = deepcopy(
                question
            )

            direct_key = (
                str(
                    candidate.get(
                        "generated_question_id",
                        "",
                    )
                    or ""
                ),
                int(
                    candidate.get(
                        "question_index",
                        index,
                    )
                    or index
                ),
            )

            rendered = (
                rendered_lookup.get(
                    direct_key
                )
            )

            if rendered is None:
                # Fall back to generated ID only.
                generated_id = direct_key[
                    0
                ]

                for lookup_key, lookup_value in rendered_lookup.items():
                    if (
                        generated_id
                        and lookup_key[
                            0
                        ]
                        == generated_id
                    ):
                        rendered = lookup_value
                        break

            if (
                rendered
                and rendered.get(
                    "notebook08_visual_status"
                )
                == "rendered"
            ):
                candidate[
                    "visual_path"
                ] = rendered.get(
                    "notebook08_visual_path"
                )

                candidate[
                    "visual_renderer"
                ] = rendered.get(
                    "notebook08_visual_renderer"
                )

                candidate[
                    "visual_asset_sha256"
                ] = rendered.get(
                    "notebook08_visual_asset_sha256"
                )

                candidate[
                    "visual_tool_name"
                ] = rendered.get(
                    "notebook08_visual_tool_name"
                )

                candidate[
                    "visual_backend"
                ] = rendered.get(
                    "notebook08_visual_backend"
                )

                candidate[
                    "visual_engine"
                ] = rendered.get(
                    "notebook08_visual_engine"
                )

                candidate[
                    "visual_architecture_phase"
                ] = "notebook_08_tool_layer"

            patched.append(
                candidate
            )

        return patched

    for field in [
        "candidate_questions",
        "questions",
    ]:
        base_manifest[
            field
        ] = patch_question_list(
            base_manifest.get(
                field,
                [],
            )
        )

    base_manifest.setdefault(
        "output_files",
        {},
    )[
        "notebook08_visual_results"
    ] = str(
        RESULTS_PATH
    )

    base_manifest.setdefault(
        "source_artifacts",
        {},
    )[
        "notebook08_visual_results"
    ] = str(
        RESULTS_PATH
    )


    base_manifest[
        "visual_integrity_gate"
    ] = results_payload.get(
        "visual_integrity_gate",
        {},
    )

    if not bool(
        (
            results_payload.get(
                "visual_integrity_gate",
                {},
            )
            or {}
        ).get(
            "release_eligible",
            False,
        )
    ):
        # Fail closed: a downstream consumer must never treat this patched
        # manifest as release-ready while a required visual is missing.
        base_manifest[
            "release_ready"
        ] = False

        base_manifest[
            "visual_release_block_reason"
        ] = (
            "One or more required Notebook 08 visuals failed validation "
            "or rendering."
        )

    base_manifest[
        "visual_tool_architecture"
    ] = {
        "renderer_notebook": (
            "08_visual_generation_tool_layer.ipynb"
        ),
        "results_schema_version": (
            NOTEBOOK08_RESULTS_SCHEMA_VERSION
        ),
        "mcp_status": "NOT_WIRED_YET",
        "separate_image_generation_api_used": False,
        "paid_ai_image_generation_used": False,
        "kroki_endpoint": KROKI_ENDPOINT,
        "renderer_policy": {
            "logic": "schemdraw",
            "technical": "kroki",
            "structured": "local_structured",
        },
        "renderer_registry": results_payload[
            "renderer_registry"
        ],
    }

    UPDATED_MANIFEST_PATH.write_text(
        json.dumps(
            base_manifest,
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        encoding="utf-8",
    )

    print(
        "Saved patched manifest copy:",
        UPDATED_MANIFEST_PATH,
    )

else:
    print(
        "Manifest patch skipped. "
        "Original Notebook 06 manifest was not modified."
    )


## 10. Optional smoke tests

Set:

```text
AGENT2_VISUAL_RUN_SMOKE_TESTS=1
```

to render a small synthetic sample covering:
- SchemDraw logic circuit;
- local truth table;
- Kroki/GraphViz network diagram;
- local array grid.

The Kroki network smoke test requires the configured Kroki endpoint to be
reachable. A Kroki failure is recorded as `render_failed` and does not cause a
fallback to a hand-positioned network diagram.


In [ ]:
if RUN_SMOKE_TESTS:
    smoke_questions = [
        {
            "question_index": 1,
            "generated_question_id": "SMOKE_LOGIC",
            "visual_requirement": "logic_gate_diagram",
            "visual": {
                "type": "logic_gate_diagram",
                "spec": {
                    "inputs": [
                        "A",
                        "B",
                        "C",
                    ],
                    "gates": [
                        {
                            "id": "g1",
                            "type": "OR",
                            "inputs": [
                                "B",
                                "C",
                            ],
                        },
                        {
                            "id": "g2",
                            "type": "AND",
                            "inputs": [
                                "A",
                                "g1",
                            ],
                        },
                    ],
                    "output": {
                        "from": "g2",
                        "label": "Q",
                    },
                    "caption": "",
                },
            },
        },
        {
            "question_index": 2,
            "generated_question_id": "SMOKE_TRUTH",
            "visual_requirement": "truth_table",
            "visual": {
                "type": "truth_table",
                "spec": {
                    "columns": [
                        "A",
                        "B",
                        "Q",
                    ],
                    "rows": [
                        [
                            0,
                            0,
                            "",
                        ],
                        [
                            0,
                            1,
                            "",
                        ],
                        [
                            1,
                            0,
                            "",
                        ],
                        [
                            1,
                            1,
                            "",
                        ],
                    ],
                    "caption": "",
                },
            },
        },
        {
            "question_index": 3,
            "generated_question_id": "SMOKE_NETWORK",
            "visual_requirement": "network_diagram",
            "visual": {
                "type": "network_diagram",
                "spec": {
                    "nodes": [
                        {
                            "id": "r1",
                            "type": "router",
                            "label": "Router",
                        },
                        {
                            "id": "s1",
                            "type": "switch",
                            "label": "Switch",
                        },
                        {
                            "id": "pc1",
                            "type": "computer",
                            "label": "PC 1",
                        },
                    ],
                    "edges": [
                        {
                            "from": "r1",
                            "to": "s1",
                            "label": "",
                        },
                        {
                            "from": "s1",
                            "to": "pc1",
                            "label": "",
                        },
                    ],
                    "caption": "",
                },
            },
        },
        {
            "question_index": 4,
            "generated_question_id": "SMOKE_ARRAY",
            "visual_requirement": "array_grid",
            "visual": {
                "type": "array_grid",
                "spec": {
                    "values": [
                        [
                            4,
                            8,
                            2,
                        ],
                        [
                            7,
                            5,
                            9,
                        ],
                    ],
                    "row_labels": [
                        "0",
                        "1",
                    ],
                    "column_labels": [
                        "0",
                        "1",
                        "2",
                    ],
                    "caption": "",
                },
            },
        },
    ]

    smoke_results = []

    for smoke_question in smoke_questions:
        smoke_result = (
            render_assessment_diagram(
                smoke_question
            )
        )

        smoke_results.append(
            smoke_result
        )

        if (
            smoke_result.get(
                "status"
            )
            == "rendered"
            and smoke_result.get(
                "path"
            )
        ):
            display(
                IPyImage(
                    filename=smoke_result[
                        "path"
                    ]
                )
            )

    print(
        json.dumps(
            smoke_results,
            indent=2,
        )
    )

else:
    print(
        "Smoke tests disabled. "
        "Set AGENT2_VISUAL_RUN_SMOKE_TESTS=1 to enable."
    )


# Final output contract — v2.2

Notebook 08 writes:

```text
OUTPUT/notebook_08_visuals/
    assets/
    notebook08_visual_results.json
    final_quiz_manifest_with_notebook08_visuals.json
```

Every required visual records its tool/backend/engine/path/hash.

### Fail-closed rule

```text
required visual
    ↓
validation
    ↓
renderer
    ↓
asset exists?
    ├─ yes → PASS
    └─ no  → visual_integrity_gate = BLOCKED
              patched manifest release_ready = false
```

Notebook 08 is intentionally not deciding pedagogy. That now happens upstream in Notebook 06 v2.38 through compatibility/relevance checks. Notebook 08 remains responsible for rendering and release blocking when a required asset is missing.


## MCP final PDF assembly

Notebook 08 remains the renderer layer. In the integrated application, the MCP adapter waits until all required visual families have completed, then rebuilds the final quiz/hybrid PDF using the rendered Notebook 08 asset paths and the current Notebook 06 PDF format.
